### 1. SET UP

In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "optuna", "-q"],
               capture_output=True)

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
sns.set_theme(context="talk", style="whitegrid", font_scale=0.85)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

print("✅ All libraries loaded.")

✅ All libraries loaded.


In [2]:
df = pd.read_csv(r'..\data\cleaned\cleaned_data.csv')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

Dataset loaded: 82,377 rows × 18 columns


,lead_time,required_car_parking_spaces,total_of_special_requests,previous_cancellations,previous_bookings_not_canceled,adr,adults,children,babies,deposit_type,market_segment,distribution_channel,customer_type,hotel,arrival_date_year,arrival_date_month,arrival_date_day_of_month,is_canceled
0,342,0,0,0,0,0.0,2,0,0,No Deposit,Direct,Direct,Transient,Resort Hotel,2015,July,1,0
1,737,0,0,0,0,0.0,2,0,0,No Deposit,Direct,Direct,Transient,Resort Hotel,2015,July,1,0
2,7,0,0,0,0,75.0,1,0,0,No Deposit,Direct,Direct,Transient,Resort Hotel,2015,July,1,0


In [3]:
TARGET = "is_canceled"

DATE_COLS   = ["arrival_date_year", "arrival_date_month", "arrival_date_day_of_month"]
SOURCE_COLS = [
    "lead_time", "required_car_parking_spaces", "total_of_special_requests",
    "previous_cancellations", "previous_bookings_not_canceled",
    "adr", "adults", "children", "babies",
    "deposit_type", "market_segment", "distribution_channel", "customer_type", "hotel",
] + DATE_COLS

### 2. Feature Engineering

We build three new engineered features from the raw source columns and reconstruct `arrival_date` for the time-based split. Raw columns needed by the baseline model are also retained in `df_eng`.

**Numerical features engineered:**

| Selected Feature | Source Columns | Transformation / Formula |
|---|---|---|
| `lead_time` | `lead_time` | Kept unchanged → StandardScaler |
| `required_car_parking_spaces` | `required_car_parking_spaces` | Kept unchanged → StandardScaler |
| `total_of_special_requests` | `total_of_special_requests` | Kept unchanged → StandardScaler |
| `prior_cancel_rate` | `previous_cancellations`, `previous_bookings_not_canceled` | Beta-smoothed historical cancel rate: `(PC + 1) / (PC + PN + 2)` |
| `adr_per_person` | `adr`, `adults`, `children`, `babies` | ADR capped at 99.5th percentile, divided by total guests: `ADR / max(guests, 1)` |
| `is_short_lead` | `lead_time` | Binary flag: `1` if `lead_time ≤ 7`, else `0` |

**Categorical features:**

| Selected Feature | Source Columns | Transformation |
|---|---|---|
| `deposit_type` | `deposit_type` | One-Hot Encoding |
| `market_segment` | `market_segment` | Rare categories → `OTHER`, then One-Hot Encoding |
| `distribution_channel` | `distribution_channel` | One-Hot Encoding |
| `customer_type` | `customer_type` | One-Hot Encoding |
| `hotel` | `hotel` | One-Hot Encoding |
| `arrival_date` | `arrival_date_year/month/day` | Reconstructed datetime for splitting only (not a model feature) |
| `is_canceled` | `is_canceled` | Target variable |

In [4]:

def engineer_features(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()

    # prior_cancel_rate ────────────────────────────────────────────────────────
    pc = out["previous_cancellations"]
    pn = out["previous_bookings_not_canceled"]
    out["prior_cancel_rate"] = (pc + 1) / (pc + pn + 2)
    print("[1] Engineered: prior_cancel_rate = (PC+1) / (PC+PN+2)")

    # adr_per_person ───────────────────────────────────────────────────────────
    adr_cap = float(out["adr"].quantile(0.995))
    out["adr_capped"] = out["adr"].clip(upper=adr_cap)
    guests = (out["adults"].fillna(0) + out["children"].fillna(0)
              + out["babies"].fillna(0)).clip(lower=1)
    out["adr_per_person"] = out["adr_capped"] / guests
    out = out.drop(columns=["adr_capped"])
    print(f"[2] Engineered: adr_per_person = adr / guests  (ADR capped at {adr_cap:.2f})")

    # is_short_lead ────────────────────────────────────────────────────────────
    out["is_short_lead"] = (out["lead_time"] <= 7).astype(int)
    pct = out["is_short_lead"].mean()
    print(f"[3] Engineered: is_short_lead = (lead_time ≤ 7)  [{pct:.1%} of bookings]")

    # market_segment: merge rare categories ────────────────────────────────────
    top10 = set(out["market_segment"].value_counts().head(10).index)
    out["market_segment"] = out["market_segment"].where(
        out["market_segment"].isin(top10), "OTHER"
    )
    print("[4] market_segment: rare categories merged → 'OTHER'")

    # Reconstruct arrival_date ──────────────────────────────────────────────────
    months = {"January":1,"February":2,"March":3,"April":4,"May":5,"June":6,
              "July":7,"August":8,"September":9,"October":10,"November":11,"December":12}
    m = out["arrival_date_month"].map(months).astype(int)
    out["arrival_date"] = pd.to_datetime({
        "year" : out["arrival_date_year"].astype(int),
        "month": m,
        "day"  : out["arrival_date_day_of_month"].astype(int),
    })
    out = out.drop(columns=DATE_COLS)
    print("[5] Reconstructed arrival_date (for splitting only, not a model feature)")

    # Retain raw columns needed for the baseline model
    BASELINE_RAWS = ["adr", "previous_cancellations", "previous_bookings_not_canceled"]

    # Final column set
    FINAL_FEATURES = [
        "lead_time", "required_car_parking_spaces", "total_of_special_requests",
        "prior_cancel_rate", "adr_per_person", "is_short_lead",
        "deposit_type", "market_segment", "distribution_channel",
        "customer_type", "hotel",
    ]
    keep = FINAL_FEATURES + BASELINE_RAWS + ["arrival_date", TARGET]
    out = out[[c for c in keep if c in out.columns]].copy()

    print(f"\nFeature engineering complete. Shape: {out.shape}")
    return out


df_eng = engineer_features(df)
df_eng.head(3)



[1] Engineered: prior_cancel_rate = (PC+1) / (PC+PN+2)
[2] Engineered: adr_per_person = adr / guests  (ADR capped at 287.00)
[3] Engineered: is_short_lead = (lead_time ≤ 7)  [21.1% of bookings]
[4] market_segment: rare categories merged → 'OTHER'


[5] Reconstructed arrival_date (for splitting only, not a model feature)

Feature engineering complete. Shape: (82377, 16)


,lead_time,required_car_parking_spaces,total_of_special_requests,prior_cancel_rate,adr_per_person,is_short_lead,deposit_type,market_segment,distribution_channel,customer_type,hotel,adr,previous_cancellations,previous_bookings_not_canceled,arrival_date,is_canceled
0,342,0,0,0.5,0.0,0,No Deposit,Direct,Direct,Transient,Resort Hotel,0.0,0,0,2015-07-01,0
1,737,0,0,0.5,0.0,0,No Deposit,Direct,Direct,Transient,Resort Hotel,0.0,0,0,2015-07-01,0
2,7,0,0,0.5,75.0,1,No Deposit,Direct,Direct,Transient,Resort Hotel,75.0,0,0,2015-07-01,0


### 3. Time-Based Split

A random split would be dishonest: the model could train on a July 2017 booking and test on a January 2016 one, the opposite of real life. Antonio et al. (2019) show that stratified random splits inflate reported performance versus a proper time-based split.

| Split | Period | Purpose |
|---|---|---|
| **Train** | 2015-07 → 2016-12 | Fit the model and learn booking patterns |
| **Validation** | 2017-01 → 2017-04 | Hyperparameter tuning & threshold selection |
| **Test** | 2017-05 → 2017-08 | Final evaluation on strictly unseen data |

In [5]:
TRAIN_END = pd.Timestamp("2016-12-31")
VAL_END   = pd.Timestamp("2017-04-30")

train = df_eng[df_eng["arrival_date"] <= TRAIN_END].copy()
val   = df_eng[(df_eng["arrival_date"] > TRAIN_END) &
               (df_eng["arrival_date"] <= VAL_END)].copy()
test  = df_eng[df_eng["arrival_date"] > VAL_END].copy()

print("=== Time-based Split Summary ===")
for name, split in [("Train", train), ("Validation", val), ("Test", test)]:
    cr = split[TARGET].mean()
    print(f"  {name:12s}: {len(split):6,} rows  "
          f"({split['arrival_date'].min().date()} → {split['arrival_date'].max().date()})  "
          f"cancel rate = {cr:.1%}")

print(f"\n  Total: {len(train)+len(val)+len(test):,} rows")



=== Time-based Split Summary ===
  Train       : 52,004 rows  (2015-07-01 → 2016-12-31)  cancel rate = 26.0%
  Validation  : 13,232 rows  (2017-01-01 → 2017-04-30)  cancel rate = 29.2%
  Test        : 17,141 rows  (2017-05-01 → 2017-08-31)  cancel rate = 35.2%

  Total: 82,377 rows


### 4. Preprocessing
- **`num`** → `StandardScaler` on 5 continuous numerical features
- **`bin`** → `passthrough` on the binary flag `is_short_lead`
- **`cat`** → `OneHotEncoder` (drop first dummy, ignore unknown categories at test time) on 5 categorical features

In [6]:
FINAL_NUMERIC      = ["lead_time", "required_car_parking_spaces",
                      "total_of_special_requests", "prior_cancel_rate", "adr_per_person"]
FINAL_BINARY       = ["is_short_lead"]
FINAL_CATEGORICAL  = ["deposit_type", "market_segment", "distribution_channel",
                      "customer_type", "hotel"]
ALL_FEATURES       = FINAL_NUMERIC + FINAL_BINARY + FINAL_CATEGORICAL


def xy(df_split: pd.DataFrame):
    """Extract feature matrix X and target y from a split."""
    return df_split[ALL_FEATURES].copy(), df_split[TARGET].copy()


def make_preprocessor():
    """
    Factory function: returns a fresh, unfitted ColumnTransformer.
    Called inside every pipeline so each model gets its own independent scaler.
    """
    return ColumnTransformer([
        ("num", StandardScaler(),                                    FINAL_NUMERIC),
        ("bin", "passthrough",                                       FINAL_BINARY),
        ("cat", OneHotEncoder(handle_unknown="ignore",
                              sparse_output=False, drop="first"),    FINAL_CATEGORICAL),
    ], remainder="drop", verbose_feature_names_out=False)


X_train, y_train = xy(train)
X_val,   y_val   = xy(val)
X_test,  y_test  = xy(test)

# Train + Val combined → used to refit final model before scoring on Test
X_fit, y_fit = xy(pd.concat([train, val], ignore_index=True))

print("Pipeline components defined.")
print(f"  X_train : {X_train.shape}  |  X_val : {X_val.shape}  |  X_test : {X_test.shape}")
print(f"  X_fit (train+val) : {X_fit.shape}")



Pipeline components defined.
  X_train : (52004, 11)  |  X_val : (13232, 11)  |  X_test : (17141, 11)
  X_fit (train+val) : (65236, 11)


### 5. Sanity Check

Before training any model, we verify the pipeline transforms correctly:
- Scaled numerical columns should have mean ≈ 0, std ≈ 1
- No NaN values in the transformed output
- Confirm the total number of output columns (OHE expands categoricals)

In [7]:
_pre = make_preprocessor()
_pre.fit(X_train)
X_train_check = _pre.transform(X_train)

# Column names after transformation
all_col_names = _pre.get_feature_names_out()
print(f"Total columns after preprocessing: {len(all_col_names)}")
print(f"  (5 scaled numerics + 1 binary + OHE-expanded categoricals)\n")

# Check scaling
num_idx   = list(range(len(FINAL_NUMERIC)))
arr_check = X_train_check[:, num_idx]
scale_df  = pd.DataFrame(arr_check, columns=FINAL_NUMERIC)
print("Scaled numerical stats (mean should be ~0, std should be ~1):")
print(scale_df.agg(["mean", "std"]).round(4).to_string())

# Check for NaN
n_nan = np.isnan(X_train_check).sum()
print(f"\nNaN count in transformed train set: {n_nan}  {'✅ Clean' if n_nan == 0 else '❌ Problem!'}")



Total columns after preprocessing: 23
  (5 scaled numerics + 1 binary + OHE-expanded categoricals)

Scaled numerical stats (mean should be ~0, std should be ~1):
      lead_time  required_car_parking_spaces  total_of_special_requests  prior_cancel_rate  adr_per_person
mean       -0.0                         -0.0                        0.0               -0.0             0.0
std         1.0                          1.0                        1.0                1.0             1.0

NaN count in transformed train set: 0  ✅ Clean


### 6. SAVE DATA

In [10]:
df.to_csv("..\data\preprocessed\preprocessed_data.csv", index=False)